# Dubbing Studio — Pipeline Notebook

Run each stage on its own cell so you can inspect and play back the artefacts as they land.

| # | Stage | Function |
|---|---|---|
| 1 | Download | `backend.downloader.download_youtube` |
| 2 | Audio extraction | `backend.media_io.extract_audio` |
| 3 | Stem separation (HTDemucs) | `backend.stem_separation.separate_stems` |
| 4 | Speaker diarization | `backend.diarize.diarize` — one of `pyannote`, `diarizen` |

Reference videos wired below — pick one via `INPUT_URL` or replace with a local path.

## 0. Setup

In [ ]:
# Add repo root to sys.path when notebook is launched from `notebooks/`.
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)-25s %(levelname)-7s %(message)s')

from IPython.display import Audio, Video, display, Markdown

DATA_DIR = REPO_ROOT / 'data'
for sub in ('videos', 'audio', 'stems', 'diarization'):
    (DATA_DIR / sub).mkdir(parents=True, exist_ok=True)

print('REPO_ROOT =', REPO_ROOT)
print('HF_TOKEN set:', bool(os.getenv('HF_TOKEN')))

In [ ]:
# Reference clips — pick one, or replace INPUT_URL with a local video path.
REFERENCE_URLS = [
    'https://www.youtube.com/watch?v=nsIMQP0QhpE',
    'https://www.youtube.com/watch?v=P26AE7NLx4Q',
    'https://www.youtube.com/watch?v=2vxQaT0Fy-c',
]

INPUT_URL = REFERENCE_URLS[1]   # change index (0/1/2) or paste your own URL/path
print('INPUT:', INPUT_URL)

## 1. Download the video

Uses `yt-dlp`. If `INPUT_URL` is a local path it will be resolved as-is.

In [ ]:
from backend.downloader import resolve_input

video_path = resolve_input(INPUT_URL, download_dir=DATA_DIR / 'videos')
print('video ->', video_path)
display(Video(str(video_path), embed=False))

## 2. Extract audio (mono 16 kHz WAV)

This is the diarization-friendly format. Stem separation runs on the source video directly, at its native sample rate.

In [ ]:
from backend.media_io import extract_audio

audio_16k = DATA_DIR / 'audio' / f'{video_path.stem}.16k.mono.wav'
extract_audio(video_path, audio_16k, sample_rate=16_000, mono=True)
print('audio ->', audio_16k)
display(Audio(str(audio_16k)))

## 3. Stem separation — dialogue vs music & effects

HTDemucs. First run downloads model weights (~80 MB) and is slow on CPU — a few minutes per minute of audio.

In [ ]:
from backend.stem_separation import separate_stems

stems = separate_stems(video_path, DATA_DIR / 'stems')
print('dialogue ->', stems.dialogue)
print('M&E      ->', stems.music_effects)

display(Markdown('**Dialogue stem (vocals)**'))
display(Audio(str(stems.dialogue)))
display(Markdown('**Music & effects stem**'))
display(Audio(str(stems.music_effects)))

## 4. Speaker diarization

Pick a model. Adapters:
- `pyannote`  — pyannote/speaker-diarization-community-1 (needs HF_TOKEN + gate accepted)
- `diarizen`  — BUT-FIT DiariZen (non-commercial weights)

First model call downloads weights.

In [ ]:
from backend.diarize import diarize
from backend.media_io import extract_audio

DIARIZATION_MODEL = 'pyannote'   # 'pyannote' | 'diarizen'

# Re-extract the dialogue stem at 16 kHz mono for the diarizer.
dia_input = DATA_DIR / 'audio' / f'{video_path.stem}.dialogue.16k.mono.wav'
extract_audio(stems.dialogue, dia_input, sample_rate=16_000, mono=True)

rttm_path = DATA_DIR / 'diarization' / f'{video_path.stem}.{DIARIZATION_MODEL}.rttm'
result = diarize(dia_input, model=DIARIZATION_MODEL, rttm_out=rttm_path)

print(f'model    : {result.model_name}')
print(f'turns    : {len(result.turns)}')
print(f'speakers : {result.num_speakers}')
print(f'rttm     : {rttm_path}')

In [ ]:
# First 20 turns.
for t in result.turns[:20]:
    print(f'{t.speaker:>7}  {t.start:8.2f}s → {t.end:8.2f}s  ({t.duration:5.2f}s)')

In [ ]:
# Play the first turn per speaker so you can hear who is who.
import soundfile as sf
import numpy as np

audio, sr = sf.read(str(dia_input))
seen = set()
for t in result.turns:
    if t.speaker in seen: continue
    seen.add(t.speaker)
    s, e = int(t.start * sr), int(t.end * sr)
    display(Markdown(f'**{t.speaker}** — {t.start:.2f}s → {t.end:.2f}s'))
    display(Audio(audio[s:e], rate=sr))

## 5. Compare all three diarization models on the same audio

Optional — expensive because each model downloads ~1-2 GB of weights and runs a full pass. Skip unless you want the head-to-head.

In [ ]:
from backend.diarize import diarize
from models.diarization import available_models

COMPARE = False    # set True to run

comparison = {}
if COMPARE:
    for m in available_models():
        try:
            r = diarize(
                dia_input, model=m,
                rttm_out=DATA_DIR / 'diarization' / f'{video_path.stem}.{m}.rttm',
            )
            comparison[m] = {'turns': len(r.turns), 'speakers': r.num_speakers}
        except Exception as e:
            comparison[m] = {'error': str(e)[:200]}

    display(Markdown('### Model comparison'))
    for m, stats in comparison.items():
        display(Markdown(f'- **{m}**: `{stats}`'))
else:
    print('COMPARE=False; flip to True to run all three models.')

## 6. Run the whole pipeline in one shot

Equivalent to `python -m backend.cli --input ... --model ...`.

In [ ]:
from backend.pipeline import run_pipeline

pr = run_pipeline(INPUT_URL, diarization_model='pyannote', data_dir=DATA_DIR)
print(pr)